In [ ]:
import numpy as np
import logging
from dataclasses import dataclass, field
from typing import Dict, Optional

In [ ]:
# Verify key imports
import sys
print(f"Python {sys.version}")

try:
    import langchain; print(f"langchain         {langchain.__version__}")
    import faiss;     print(f"faiss-cpu         installed")
    import redis;     print(f"redis             {redis.__version__}")
    import fastapi;   print(f"fastapi           {fastapi.__version__}")
    import pydantic;  print(f"pydantic          {pydantic.__version__}")
    print("\n✅  All dependencies available")
except ImportError as e:
    print(f"\n⚠️  Missing: {e}  — run the pip installs above")


In [ ]:
logging.basicConfig(level = logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger= logging.getLogger(__name__)

In [ ]:
## seting up the data mmodels 
@dataclass
class Document:
    id: str
    content: str
    metadata: dict = field(default_factory=dict)
    embedding: Optional[list[float]]=None

@dataclass
class RetrievedChunk:
    document: Document
    score:float
    relevance_score: float=0.0 ## this is set by the retrieval validation agent
    chunk_index: int=0

@dataclass
class Claim:
    text:str
    supported: bool =False
    supporting_chunks: list[RetrievedChunk] = field(default_factory = list)
    confidence: float = 0.0

@dataclass
class RAGResponse:
    query: str
    answer:str
    claims:list[Claim]
    retrieved_chunks: list[RetrievedChunk]
    confidence_score: float
    hallucination_risk:str ## LOW/MEDIUM/HIGH
    latency_ms: float
    agent_trace: list[dict] = field(default_factory=list)


Things to do:
- compute the embedding
- cosine similarity
- setting up the vectorstore
- accesing the base agent through API


In [ ]:
## (A@B)/(||A||*||B||)

def cosine_similarities(a:np.array, b:np.array):
    assert a.
    a, b = np.array(a), np.array(b)
    dot_ab = np.dot(a,b)
    mag_a = np.sqrt(np.sum(np.square(a)))
    mag_b = np.sqrt(np.sum(np.square(b)))
    sum_mag_ab = np.add(mag_a, mag_b)

    cosine_sim = dot_ab/sum_mag_ab

    return cosine_sim


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
a = [5,6,3,5,6,7,8]
b = [3,4,5,6,74,3,6]

a_arr = np.array([[a]])
b_arr = np.array([[b]])

cosine = cosine_similarity(a, b)
print(cosine)
cos_sklearn = cosine_similarity(a_arr, b_arr)

In [ ]:
import qdrant

In [ ]:
import os

In [ ]:
!pwd

In [ ]:
from multiagent_rag_system.src.embedding.embedding import get_embedder

In [1]:
from multiagent_rag_system.agent.agents.retrieval_agent import ChunkRetrieval
from multiagent_rag_system.agent.agents.reranker_agent import RerankerAgent
from multiagent_rag_system.agent.agents.consensus_agent import ConsensusAgent
from multiagent_rag_system.agent.agents.claim_verification_agent import ClaimVerificationAgent
from multiagent_rag_system.agent.agents.confidence_score_agent import ConfidenceScoringAgent
from multiagent_rag_system.agent.agents.query_expansion import QueryExpansionAgent
from multiagent_rag_system.agent.agents.evaluator import RAGASEvaluator
from multiagent_rag_system.agent.pipeline.pipeline import RAGOrchestrator
from multiagent_rag_system.agent.agents.doc_ingestion import DocumentIngestionPipeline

/home/emmanuel/Desktop/Multiagent_RAG_system/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Text = """
A vector store is a specialized system designed to store, index, and search high-dimensional numerical representations (embeddings) of data like text, images, and audio, enabling similarity-based retrieval rather than exact keyword matching.  While often used interchangeably with "vector database," a vector store is typically a lighter library or feature focused on core storage and querying, whereas a vector database provides full database capabilities like distributed scaling, security, and complex query management. 

These systems are the foundation of modern AI infrastructure, powering critical applications such as Retrieval-Augmented Generation (RAG), semantic search, and real-time recommendation engines.  They operate by converting data into vectors that capture semantic meaning, allowing AI models to find conceptually similar items in milliseconds using Approximate Nearest Neighbor (ANN) algorithms like HNSW or IVF. 

Key Characteristics and Functionality
Data Representation: Converts unstructured data into fixed-length numerical arrays (vectors) where related items are positioned close together in high-dimensional space. 
Search Mechanism: Uses similarity metrics like cosine similarity, Euclidean distance, or dot product to rank results by relevance rather than exact string matches. 
Performance: Optimized for high-speed retrieval over billions of data points, making them essential for production-scale AI applications. 
Types and Architectures
Vector stores are generally categorized into three main types based on their architecture and use cases:

Native Vector Databases: Tools built specifically for AI workloads (e.g., Pinecone, Qdrant, Weaviate, Milvus) offering superior raw retrieval performance and advanced indexing. 
Hybrid Databases: Traditional databases that have added vector capabilities (e.g., PostgreSQL with pgvector, MongoDB, Elasticsearch), providing easier integration with existing infrastructure.
Cloud-Managed Services: Fully managed solutions (e.g., Amazon Bedrock, Azure AI Search, Google Vertex AI) that handle scaling and operations but may involve vendor lock-in.
Comparison: Vector Store vs. Vector Database
Feature	Vector Store	Vector Database
Scope	Lightweight library or extension	Standalone, robust system
Primary Focus	Fast storage and similarity search	Full data management (CRUD, security)
Scalability	Often limited or in-memory	Distributed, petabyte-scale support
Use Case	Prototyping, experimentation, simple apps	Enterprise-grade, production, complex queries
Persistence	Minimal or memory-based	Durable, fault-tolerant storage
Common Use Cases
Vector stores are indispensable for industries leveraging AI to manage unstructured data, including:

Semantic Search: Finding content based on meaning (e.g., "documents similar to this query") rather than keywords. 
Recommendation Systems: Powering personalized suggestions on platforms like Amazon and Netflix by matching user and item vectors. 
LLM Memory: Providing long-term context for Large Language Models to ground answers in proprietary data and reduce hallucinations. 
Fraud Detection: Identifying anomalies in transaction patterns by analyzing vector distances in financial data. 
Multi-modal Search: Retrieving images, audio, or text based on cross-modal similarity (e.g., finding images matching a text description). 

"""

In [3]:
expander = QueryExpansionAgent()
retrieve= ChunkRetrieval()
rerank= RerankerAgent()
gen = ConsensusAgent()
verifier = ClaimVerificationAgent()
confidence = ConfidenceScoringAgent()
evaluate = RAGASEvaluator()
ingestion = DocumentIngestionPipeline()

from multiagent_rag_system.src.models.models import QueryRequest
from multiagent_rag_system.src.models.models import IngestRequest
ingestion_request = IngestRequest(content=Text, metadata={})

In [ ]:
await ingestion.ingest_text(ingestion_request)

In [4]:
query=QueryRequest(query="What is vector database", filters={})

In [5]:
expanded_query, _ = await expander.expand(query=query)
expanded_query

['What is vector database',
 'A vector database is a specialized data store designed to index, store, and retrieve high‑dimensional numeric vectors—often generated as embeddings from machine‑learning models—based on similarity metrics such as cosine similarity or Euclidean distance. It employs approximate nearest‑neighbor (ANN) algorithms and indexing structures (e.g., HNSW, IVF‑PQ, or ANNOY) to enable sub‑millisecond latency for similarity search over billions of vectors. Unlike traditional relational databases that excel at exact match queries, vector databases support semantic search, recommendation, and anomaly‑detection workloads by efficiently handling vector arithmetic and distance calculations at scale.',
 'Can you explain what a vector database is?',
 'How does a vector database differ from conventional database systems?',
 'What purpose does a vector database serve in AI and similarity search?']

In [6]:
retrieved_chunk, _ = await retrieve.retrieve(expanded_query)
retrieved_chunk

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 174.30it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[RetrievedChunk(chunk=DocumentChunk(id='9789fc5e-46c5-4747-9514-0dd18a9357bf', doc_id='c7d415b0-dfeb-4fb3-b268-9c22b0abb90c', content='A vector store is a specialized system designed to store, index, and search high-dimensional numerical representations (embeddings) of data like text, images, and audio, enabling similarity-based retrieval rather than exact keyword matching. While often used interchangeably with "vector database," a vector store is typically a lighter library or feature focused on core storage and querying, whereas a vector database provides full database capabilities like distributed scaling, security, and complex query', metadata={}, chunk_index=0, embedding=None, page_number=None, created_at=datetime.datetime(2026, 3, 28, 14, 30, 56, 534622, tzinfo=datetime.timezone.utc)), vector_score=0.794110751221745, relevance_score=0.0),
 RetrievedChunk(chunk=DocumentChunk(id='971278b4-fbac-4647-87b8-4ad17b0e798d', doc_id='c7d415b0-dfeb-4fb3-b268-9c22b0abb90c', content='Data Rep

In [7]:
re, even = await rerank.rerank(query.query, chunks = retrieved_chunk)

Loading weights: 100%|██████████| 393/393 [00:14<00:00, 26.80it/s]


In [8]:
re

[RerankedChunk(chunk=DocumentChunk(id='9789fc5e-46c5-4747-9514-0dd18a9357bf', doc_id='c7d415b0-dfeb-4fb3-b268-9c22b0abb90c', content='A vector store is a specialized system designed to store, index, and search high-dimensional numerical representations (embeddings) of data like text, images, and audio, enabling similarity-based retrieval rather than exact keyword matching. While often used interchangeably with "vector database," a vector store is typically a lighter library or feature focused on core storage and querying, whereas a vector database provides full database capabilities like distributed scaling, security, and complex query', metadata={}, chunk_index=0, embedding=None, page_number=None, created_at=datetime.datetime(2026, 3, 28, 14, 30, 56, 534622, tzinfo=datetime.timezone.utc)), similarity_score=0.794110751221745, reranker_score=0.9980668425559998),
 RerankedChunk(chunk=DocumentChunk(id='f62f6adc-ed8a-4e61-a212-7d2aeb87ca18', doc_id='c7d415b0-dfeb-4fb3-b268-9c22b0abb90c', c

In [17]:
gen = ConsensusAgent()

In [19]:
generated, answer, _ = await gen.run(query.query, re)

In [23]:
for a in answer:
    print(a)

A vector database is a full‑featured data management system that stores, indexes, and retrieves high‑dimensional embeddings while providing traditional database capabilities such as durable storage, CRUD operations, security, and distributed, petabyte‑scale scalability for production‑grade workloads【1†L1-L3】【3†L2-L5】
A vector database is a full‑featured data management system that stores, indexes, and retrieves high‑dimensional embeddings while providing traditional database capabilities such as durable storage, CRUD operations, security, and distributed, petabyte‑scale scalability for production‑grade workloads【1†L1-L3】【3†L2-L5】
A vector database is a full‑featured data management system that stores, indexes, and retrieves high‑dimensional embeddings while providing traditional database capabilities such as durable storage, CRUD operations, security, and distributed, petabyte‑scale scalability for production‑grade workloads【1†L1-L3】【3†L2-L5】
A vector database is a standalone, robust s

In [ ]:
from multiagent_rag_system.src.llm.llms import get_llm_client

In [ ]:
llm = get_llm_client()

In [ ]:
llm.complete()

In [ ]:
from multiagent_rag_system.src.llm.llms import get_llm_client
llm = get_llm_client()

In [ ]:
llm.complete()

In [ ]:
response = await llm.complete("I'm Emmanuel", "who are you")

In [ ]:
from multiagent_rag_system.src.utils.config_loader import get_settings
settings= get_settings()
settings.openai_api_key

In [ ]:
generated, _, =await gen.run(query.query, re)

In [ ]:
gen, _ = await gen.run(query.query, re)

In [ ]:
SYSTEM_PROMPT = """You are a precise, citation-grounded 
                question-answering assistant
                
                Rules you MUST follow:
                1. Answer only from the provided context passages.
                2. If the answer is not in the context, say: "The provided sources do not contain enough information to answer this question."
                3. Do NOT invent facts, statistics, names, or dates not present in the sources.
                4. Reference source numbers (e.g. [1], [2]) inline when using that source.
                5. Keep the answer concise: 2-4 sentences for simple questions, structured paragraphs for complex ones.
                6. Never Start with "Based on the context" - be direct.
                """

In [ ]:
content = []
for i,re in enumerate(re):
    context_parts.append(
        f"[{i}] {re.chunk.content.strip()}"
    )

In [ ]:
from multiagent_rag_system.src.llm.llms import get_llm_client
llm = get_llm_client()

In [ ]:
data = await llm.complete("I am Emmanuel", "who are you", "my name is ajala emmanuel")

In [ ]:
data.text

In [ ]:
content.append(SYSTEM_PROMPT)

In [ ]:
context_parts = []
context_parts.append(SYSTEM_PROMPT)

In [ ]:
print(context_parts)

In [ ]:
merfed, event = await retrieve.retrieve(["What is your name"])

In [ ]:
pipeline = RAGOrchestrator()

In [ ]:
response = await pipeline.run(query)

In [ ]:
response

In [ ]:
embed.embed(["lhkehfdkfefekga"])

In [ ]:
get_embedder().embed(['ldlfdheffdfhal'])